## Prerequisites

Runtime: Python 3, T4 GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# InternVL3 works with stable transformers (>=4.37.2) via trust_remote_code.
# Pin to 4.46.3 to avoid the init_empty_weights() regression in the source build
# (which causes "Tensor.item() cannot be called on meta tensors" in InternVL's __init__).
# ⚠️ Restart the runtime after this cell if another notebook already installed transformers from source.
%pip install -q "transformers==4.46.3" accelerate torchvision Pillow

In [ ]:
import torch
import torchvision.transforms as T
from PIL import Image
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer

### Image preprocessing helpers

InternVL uses a custom dynamic-tiling strategy: the image is split into up to
`max_num` tiles of 448×448 px, plus an optional thumbnail. These helpers are
taken verbatim from the official InternVL3 documentation.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def build_transform(input_size):
    return T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float('inf')
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=6, image_size=448, use_thumbnail=True):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height
    target_ratios = set(
        (i, j)
        for n in range(min_num, max_num + 1)
        for i in range(1, n + 1)
        for j in range(1, n + 1)
        if min_num <= i * j <= max_num
    )
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])
    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size
    )
    target_width  = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]
    resized_img = image.resize((target_width, target_height))
    processed_images = []
    for i in range(blocks):
        box = (
            (i % (target_width // image_size)) * image_size,
            (i // (target_width // image_size)) * image_size,
            ((i % (target_width // image_size)) + 1) * image_size,
            ((i // (target_width // image_size)) + 1) * image_size,
        )
        processed_images.append(resized_img.crop(box))
    if use_thumbnail and len(processed_images) != 1:
        processed_images.append(image.resize((image_size, image_size)))
    return processed_images

def load_image(image_file, input_size=448, max_num=6):
    image = Image.open(image_file).convert('RGB')
    transform = build_transform(input_size=input_size)
    tiles = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = torch.stack([transform(tile) for tile in tiles])
    return pixel_values

In [ ]:
from pathlib import Path

WORKING_DIR = Path('/content/drive/MyDrive/aiOCR')
MODEL_NAME = 'OpenGVLab/InternVL3-2B'

# InternVL3-2B is ~1.8B params → ~3.6 GB in fp16, well within T4's 15 GB.
# BitsAndBytesConfig (4-bit or 8-bit) always calls accelerate's init_empty_weights()
# internally, which creates meta tensors that conflict with InternVL's custom __init__
# (it calls torch.linspace(...).item() there). Skipping BnB avoids the issue entirely.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, use_fast=False)
model = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    use_flash_attn=False,       # Flash Attention 2 requires Ampere+ (A100/RTX30xx); T4 is Turing
    trust_remote_code=True,
).eval().cuda()

In [ ]:
IMAGE_FILE = WORKING_DIR / 'images/pineda1/pineda1_page_3.png'

## Inference

In [ ]:
import time

image_stem   = IMAGE_FILE.stem
image_folder = IMAGE_FILE.parent.name

# max_num=6 → at most 6 tiles + 1 thumbnail = 7 forward passes through ViT; safe on T4
pixel_values = load_image(IMAGE_FILE, input_size=448, max_num=6).to(torch.float16).cuda()

question = (
    '<image>\n'
    'Convert the document to plain text, as close to the original as possible '
    '(including typos, print errors, and original grammar and spelling). '
    'Do not add any formatting, markdown, or annotations.'
)

# Greedy decoding (do_sample=False): avoids torch.multinomial entirely.
# do_sample=True + fp16 causes NaN/Inf probabilities on T4 → CUDA device-side assert.
# Greedy is also more deterministic and reproducible for OCR benchmarking.
generation_config = dict(
    max_new_tokens=4096,
    do_sample=False,
)

t0 = time.time()
with torch.no_grad():
    transcription = model.chat(tokenizer, pixel_values, question, generation_config)
elapsed = time.time() - t0

print(f'Done in {elapsed:.1f}s')
print(transcription)


### Saving the output

In [ ]:
# Transcription → transcriptions/InternVL3-2B/<stem>.md
transcription_out = WORKING_DIR / 'transcriptions/InternVL3-2B'
transcription_out.mkdir(parents=True, exist_ok=True)
(transcription_out / f'{image_stem}.md').write_text(transcription, encoding='utf-8')
print(f'Saved: transcriptions/InternVL3-2B/{image_stem}.md')